# Perseus MCP Latin Workflow: Augustine's *Epistulae*

This notebook demonstrates a Latin-first MCP workflow using Augustine's selected letters in the live Perseus CTS inventory.

## Table of contents

1. Understand the language option
2. Load the local MCP server
3. Verify language-aware tool schemas
4. Discover Augustine in the Latin inventory
5. Select an advertised Latin edition
6. Navigate references and retrieve a passage
7. Run a small Latin text analysis
8. Try a Latin Scaife search
9. Review limits and provenance

## 1 - Understand the language option

The MCP tools use language in two related ways:

- `list_text_groups`, `find_author_names`, `get_author_resources`, and `get_work_resources` use `language="latin"` as a real CTS work-language filter;
- `search_perseus`, `search_within_text`, and `get_passage_highlights` use it to choose query normalization. Latin input is preserved instead of being interpreted as Greek Beta Code;
- passage and navigation tools do not need a separate language argument because the CTS URN already contains a `latinLit` or `greekLit` namespace and a specific edition.

## 2 - Load the MCP server

Open this notebook from inside the repository when using the default `"repo"` source. The first cell installs Perseus MCP into the active Jupyter kernel; set `PERSEUS_MCP_INSTALL_SOURCE` to `"repo"` for editable development-branch work or `"pypi"` for the published package. The setup locates `src/perseus_mcp/server.py` when available, points metadata caching at a stable cache directory, and connects to the FastMCP object in-process.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Use "repo" while working on this development checkout.
# Use "pypi" to run against the published package normal users install.
PERSEUS_MCP_INSTALL_SOURCE = "repo"  # "repo" or "pypi"
PERSEUS_MCP_PYPI_SPEC = "perseus-mcp"

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "pyproject.toml").exists()
        and (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

install_source = PERSEUS_MCP_INSTALL_SOURCE.lower()
if install_source == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find the Perseus-mcp repository from {START}. Open this notebook inside the repository checkout or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    package_target = ["--editable", str(REPO_ROOT)]
    package_label = f"editable repository at {REPO_ROOT}"
elif install_source == "pypi":
    package_target = ["--force-reinstall", PERSEUS_MCP_PYPI_SPEC]
    package_label = PERSEUS_MCP_PYPI_SPEC
else:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *package_target,
        "python-dotenv>=1.0.0",
    ]
)

print(f"Installed perseus-mcp from {package_label} into this kernel")

In [ ]:
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
import importlib
import json
import os
import re
import sys

from fastmcp import Client

START = Path.cwd().resolve()
REPO_ROOT = next(
    (
        candidate
        for candidate in [START, *START.parents]
        if (candidate / "src" / "perseus_mcp" / "server.py").exists()
    ),
    None,
)

PERSEUS_MCP_INSTALL_SOURCE = globals().get("PERSEUS_MCP_INSTALL_SOURCE", "repo").lower()
if PERSEUS_MCP_INSTALL_SOURCE not in {"repo", "pypi"}:
    raise ValueError("PERSEUS_MCP_INSTALL_SOURCE must be 'repo' or 'pypi'")

if PERSEUS_MCP_INSTALL_SOURCE == "repo":
    if REPO_ROOT is None:
        raise RuntimeError(
            f"Could not find src/perseus_mcp/server.py from {START}. Open this notebook inside the Perseus-mcp repository or set PERSEUS_MCP_INSTALL_SOURCE = 'pypi'."
        )
    SRC_DIR = REPO_ROOT / "src"
    if str(SRC_DIR) not in sys.path:
        sys.path.insert(0, str(SRC_DIR))
elif REPO_ROOT is not None:
    SRC_DIR = REPO_ROOT / "src"
    src_dir_resolved = SRC_DIR.resolve()

    def _is_repo_src(path_entry):
        try:
            return Path(path_entry).resolve() == src_dir_resolved
        except (OSError, RuntimeError):
            return False

    sys.path = [path_entry for path_entry in sys.path if not _is_repo_src(path_entry)]

if "load_dotenv" in globals():
    if REPO_ROOT is not None:
        load_dotenv(REPO_ROOT / ".env", override=False)
    else:
        load_dotenv(override=False)

CACHE_ROOT = REPO_ROOT if REPO_ROOT is not None else START
os.environ.setdefault(
    "PERSEUS_MCP_CACHE_DIR",
    str(CACHE_ROOT / ".cache" / "perseus-mcp"),
)

for module_name in [
    name
    for name in list(sys.modules)
    if name == "perseus_mcp" or name.startswith("perseus_mcp.")
]:
    del sys.modules[module_name]

from fastmcp import Client
from perseus_mcp import server

server = importlib.reload(server)
mcp = server.mcp

print(f"Install source: {PERSEUS_MCP_INSTALL_SOURCE}")
print(f"Repository root: {REPO_ROOT if REPO_ROOT is not None else 'not found'}")
print(f"Cache directory: {os.environ['PERSEUS_MCP_CACHE_DIR']}")
print(f"Loaded MCP server: {mcp.name}")

In [3]:
def tool_text(result):
    return "\n".join(
        block.text
        for block in result.content
        if getattr(block, "text", None) is not None
    )


async def call_json(client, tool_name, arguments=None):
    result = await client.call_tool(tool_name, arguments or {})
    return json.loads(tool_text(result))


def tool_properties(tool):
    schema = getattr(tool, "inputSchema", None) or getattr(tool, "input_schema", {})
    return (schema or {}).get("properties", {})

## 3 - Verify language-aware tool schemas

This drift check confirms that all discovery and search tools used below expose a `language` argument.

In [4]:
language_aware_tools = [
    "list_text_groups",
    "find_author_names",
    "get_author_resources",
    "get_work_resources",
    "search_perseus",
    "search_within_text",
    "get_passage_highlights",
]

async with Client(mcp) as client:
    tools = {tool.name: tool for tool in await client.list_tools()}

schema_summary = {
    name: sorted(tool_properties(tools[name]))
    for name in language_aware_tools
}
assert all("language" in properties for properties in schema_summary.values())
schema_summary

{'list_text_groups': ['language', 'limit', 'query'],
 'find_author_names': ['language', 'limit', 'query'],
 'get_author_resources': ['author', 'language'],
 'get_work_resources': ['language', 'urn_or_title'],
 'search_perseus': ['author',
  'language',
  'page_num',
  'preserve_operators',
  'query',
  'query_format',
  'result_format',
  'search_kind',
  'text_group',
  'work'],
 'search_within_text': ['language',
  'offset',
  'preserve_operators',
  'query',
  'query_format',
  'search_kind',
  'size',
  'text_urn'],
 'get_passage_highlights': ['language',
  'passage_urn',
  'preserve_operators',
  'query',
  'query_format',
  'search_kind']}

## 4 - Discover Augustine in the Latin inventory

Author discovery is constrained to Latin works. The assertion checks identity and language, not a fixed inventory size.

In [5]:
async with Client(mcp) as client:
    author_matches = await call_json(
        client,
        "find_author_names",
        {"query": "Augustine", "language": "latin", "limit": 10},
    )

assert author_matches["language"] == "lat"
assert author_matches["match_count"] >= 1
assert any("Augustine" in " ".join(item["names"]) for item in author_matches["authors"])
author_matches

{'query': 'Augustine',
 'language': 'lat',
 'match_count': 1,
 'authors': [{'urn': 'urn:cts:latinLit:stoa0040',
   'names': ['Augustine, Saint'],
   'matched_names': ['Augustine, Saint'],
   'works_count': 1,
   'works': [{'urn': 'urn:cts:latinLit:stoa0040.stoa0011',
     'language': 'lat',
     'titles': ['Epistualae']}]}]}

## 5 - Select an advertised Latin edition

The live inventory currently labels the work title `Epistualae` and the edition `Epistulae. Selections.`. The code selects by advertised language and resource type rather than depending on that spelling.

In [6]:
async with Client(mcp) as client:
    augustine = await call_json(
        client,
        "get_author_resources",
        {"author": "Augustine", "language": "latin"},
    )

authors = [
    item for item in augustine["authors"]
    if item.get("urn") == "urn:cts:latinLit:stoa0040"
]
assert authors, "The advertised Augustine textgroup was not found"

latin_works = [work for work in authors[0]["works"] if work.get("language") == "lat"]
assert latin_works
work = latin_works[0]
edition = next(
    resource for resource in work["editions"]
    if ".perseus-lat" in resource.get("urn", "")
)

selection = {
    "author_urn": authors[0]["urn"],
    "author_names": authors[0]["names"],
    "work_urn": work["urn"],
    "work_titles": work["titles"],
    "edition_urn": edition["urn"],
    "edition_label": edition.get("label"),
}
selection

{'author_urn': 'urn:cts:latinLit:stoa0040',
 'author_names': ['Augustine, Saint'],
 'work_urn': 'urn:cts:latinLit:stoa0040.stoa0011',
 'work_titles': ['Epistualae'],
 'edition_urn': 'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1',
 'edition_label': 'Epistulae. Selections.'}

In [7]:
async with Client(mcp) as client:
    work_lookup = await call_json(
        client,
        "get_work_resources",
        {"urn_or_title": work["urn"], "language": "latin"},
    )

assert work_lookup["language"] == "lat"
assert work_lookup["match_count"] >= 1
assert all(match["work"]["language"] == "lat" for match in work_lookup["matches"])
work_lookup

{'query': 'urn:cts:latinLit:stoa0040.stoa0011',
 'language': 'lat',
 'match_count': 1,
 'matches': [{'author': {'urn': 'urn:cts:latinLit:stoa0040',
    'names': ['Augustine, Saint'],
    'works_count': 1},
   'work': {'urn': 'urn:cts:latinLit:stoa0040.stoa0011',
    'language': 'lat',
    'titles': ['Epistualae'],
    'editions': [{'type': 'edition',
      'urn': 'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1',
      'label': 'Epistulae. Selections.',
      'description': 'Select Letters. Augustine, Saint. James Houston Baxter. William Heinemann Ltd.; Harvard University Press. London; Cambridge, Massachusetts. 1930. Keyboarding.'}],
    'translations': [],
    'resources': [{'type': 'edition',
      'urn': 'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1',
      'label': 'Epistulae. Selections.',
      'description': 'Select Letters. Augustine, Saint. James Houston Baxter. William Heinemann Ltd.; Harvard University Press. London; Cambridge, Massachusetts. 1930. Keyboarding.'}]}}]}

## 6 - Navigate references and retrieve a passage

Augustine's letters advertise a two-level citation scheme: letter and section. We request level-two references, which can still include a whole-letter reference where no section is exposed, then choose the first advertised passage URN and retrieve readable Latin text.

In [8]:
async with Client(mcp) as client:
    references = await call_json(
        client,
        "get_valid_references_json",
        {"urn": edition["urn"], "level": 2, "limit": 8, "offset": 0},
    )

assert references["references"]
assert all(ref.startswith(edition["urn"] + ":") for ref in references["references"])
passage_urn = references["references"][0]
references

{'urn': 'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1',
 'level': 2,
 'total_count': 274,
 'offset': 0,
 'limit': 8,
 'returned_count': 8,
 'has_more': True,
 'references': ['urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:1',
  'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:2.1',
  'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:2.2',
  'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:3.1',
  'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:3.2',
  'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:3.3',
  'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:4.1',
  'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:4.2']}

In [9]:
async with Client(mcp) as client:
    passage_result = await client.call_tool(
        "get_passage_plaintext",
        {"urn": passage_urn},
    )

latin_text = tool_text(passage_result).strip()
assert latin_text
print(passage_urn)
print()
print(latin_text[:3000])

urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:1

Bene inter nos convenit, ut opinor, omnia, quae corporeus sensus adtingit, ne puncto quidem temporis eodem modo manere posse, sed labi, effluere et praesens nihil obtinere, id est, ut latine loquar, non esse. Horum itaque amorem perniciosissimum poenarumque plenissimum vera et divina philosophia monet frenare atque sopire, ut se toto animus, etiam dum hoc corpus agit, in ea, quae semper eiusdem modi sunt neque peregrino pulchro placent, feratur atque aestuet. Quae cum ita sint et cum te verum ac simplicem, qualis sine ulla sollicitudine amari potes, in semet ipsa mens videat, fatemur tamen congressum istum atque conspectum tuum, cum a nobis corpore discedis locisque seiungeris, quaerere nos eoque, dum licet, cupere fratribus. Quod profecto vitium, si te bene novi, amas in nobis et, cum omnia bona optes carissimis et familiarissimis tuis, ab hoc eos sanare metuis. Si autem tam potenti animo es, ut et agnoscere hunc laqueum et eo captos 

## 7 - Run a small Latin text analysis

This deliberately simple token count is transparent and reproducible. It is not lemmatization: inflected forms remain separate tokens.

In [10]:
tokens = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]+", latin_text.casefold())
token_counts = Counter(tokens)

analysis = {
    "passage_urn": passage_urn,
    "characters": len(latin_text),
    "tokens": len(tokens),
    "unique_tokens": len(token_counts),
    "most_common": token_counts.most_common(15),
}
assert analysis["tokens"] > 0
analysis

{'passage_urn': 'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:1',
 'characters': 1491,
 'tokens': 232,
 'unique_tokens': 181,
 'most_common': [('et', 8),
  ('ut', 6),
  ('cum', 5),
  ('non', 4),
  ('atque', 4),
  ('in', 4),
  ('te', 4),
  ('si', 4),
  ('quae', 3),
  ('dum', 3),
  ('quod', 3),
  ('bene', 2),
  ('nos', 2),
  ('omnia', 2),
  ('ne', 2)]}

## 8 - Try a Latin Scaife search

For Latin, use `language="latin"` so an ASCII query such as `deus` is preserved. The author argument resolves through CTS and may become a Scaife textgroup scope. CTS and Scaife inventories are separate, so zero results can reflect index coverage rather than absence from Augustine.

In [11]:
async with Client(mcp) as client:
    latin_search = await call_json(
        client,
        "search_perseus",
        {
            "query": "deus",
            "language": "latin",
            "query_format": "unicode",
            "author": "Augustine",
            "search_kind": "form",
            "page_num": 1,
            "result_format": "instances",
        },
    )

search_summary = {
    "query": latin_search.get("q"),
    "total_count": latin_search.get("total_count"),
    "returned_results": len(latin_search.get("results") or []),
    "author_scope": latin_search.get("author_scope"),
}
search_summary

{'query': 'deus',
 'total_count': 2183,
 'returned_results': 10,
 'author_scope': {'query': 'Augustine',
  'match_count': 1,
  'text_group': 'urn:cts:latinLit:stoa0040',
  'note': 'Author scope was sent to Scaife as a server-side text_group filter.'}}

## 9 - Review limits and provenance

- CTS inventory, editions, references, and passage text are live upstream data and can change.
- The selected Perseus edition contains only selected letters; it is not Augustine's complete correspondence.
- The simple token count does not normalize spelling, remove stop words, or identify lemmas.
- Scaife search coverage may differ from the Perseus CTS inventory.
- Record the execution date, exact URNs, complete tool arguments, and retrieved text when using the result in research.

The essential workflow is: filter inventory by Latin, discover rather than guess URNs, select an advertised edition, navigate valid references, retrieve text, and keep search/index claims separate from textual claims.

In [12]:
provenance = {
    "executed_at_utc": datetime.now(timezone.utc).isoformat(),
    "language": "latin",
    **selection,
    "passage_urn": passage_urn,
    "reference_level": 2,
    "search_query": "deus",
}
provenance

{'executed_at_utc': '2026-06-18T20:04:09.820830+00:00',
 'language': 'latin',
 'author_urn': 'urn:cts:latinLit:stoa0040',
 'author_names': ['Augustine, Saint'],
 'work_urn': 'urn:cts:latinLit:stoa0040.stoa0011',
 'work_titles': ['Epistualae'],
 'edition_urn': 'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1',
 'edition_label': 'Epistulae. Selections.',
 'passage_urn': 'urn:cts:latinLit:stoa0040.stoa0011.perseus-lat1:1',
 'reference_level': 2,
 'search_query': 'deus'}

## Sources and notebook version

This notebook uses the local MCP implementation in [`src/perseus_mcp/server.py`](../src/perseus_mcp/server.py), the live Perseus CTS inventory and passage service, the live Scaife search service, and FastMCP's in-process client.

| Field | Value |
|---|---|
| Author | Tony Jurg |
| Version | 1.0 |
| Date | June 18, 2026 |